In [26]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from scipy.fftpack import fft, ifft

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson
import scipy.signal as signal

from scipy.signal import welch
from scipy.integrate import simpson
from fooof import FOOOF
from fooof.sim.gen import gen_aperiodic
from fooof.plts.spectra import plot_spectra
from fooof.plts.annotate import plot_annotated_peak_search
from fooof import FOOOFGroup

subject 92and 102 left out for now

compute PSI and PAC

In [27]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""
CFG_YAML_freq = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_bandpass_{}_preprocessed_combined_py_test.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config(config=CFG_YAML):
    cfg = OmegaConf.create(yaml.safe_load(config))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [28]:
def calculate_modulation_index(phase_signal, amplitude_signal, n_bins=18):
    # Extract phase using Hilbert transform
    phase = np.angle(signal.hilbert(phase_signal))
    
    # Extract amplitude envelope
    amplitude_envelope = np.abs(signal.hilbert(amplitude_signal))
    
    # Create phase bins
    phase_bins = np.linspace(-np.pi, np.pi, n_bins+1)
    
    # Calculate mean amplitude in each phase bin
    mean_amplitude = []
    for i in range(n_bins):
        indices = np.logical_and(phase >= phase_bins[i], phase < phase_bins[i+1])
        mean_amplitude.append(np.mean(amplitude_envelope[indices]))
    
    # Normalize the mean amplitude
    mean_amplitude = np.array(mean_amplitude) / np.sum(mean_amplitude)
    
    # Calculate Kullback-Leibler divergence from uniform distribution
    uniform = np.ones(n_bins) / n_bins
    mi = np.sum(mean_amplitude * np.log(mean_amplitude / uniform))
    
    return mi

In [29]:
def calculate_psi(signal_x, signal_y, fs, freq_range, filter_order=4):
    """
    Calculate Phase Synchronization Index
    
    Parameters:
    signal_x, signal_y: 1D arrays
    fs: sampling frequency
    freq_range: tuple of (low_freq, high_freq)
    filter_order: order of the bandpass filter
    
    Returns:
    psi: Phase Synchronization Index value
    """
    #low = freq_range[0]
    #high = freq_range[1]
    # Bandpass filter signals to frequency range of interest
    #low, high = freq_range
    #b, a = signal.butter(filter_order, [low/(fs/2), high/(fs/2)], btype='band')
    #x_filtered = signal.filtfilt(b, a, signal_x)
    #y_filtered = signal.filtfilt(b, a, signal_y)

    #x_filtered = mne.time_frequency.psd_array_multitaper(signal_x, 1000, fmin=low, fmax=high, adaptive=True, low_bias=True, normalization='full', verbose=False)
    #y_filtered = mne.time_frequency.psd_array_multitaper(signal_y, 1000, fmin=low, fmax=high, adaptive=True, low_bias=True, normalization='full', verbose=False)
    
    # Extract instantaneous phase using Hilbert transform
    phase_x = np.angle(signal.hilbert(signal_x))
    phase_y = np.angle(signal.hilbert(signal_y))
    
    # Calculate phase difference
    phase_diff = phase_x - phase_y
    
    # Calculate PSI using circular statistics
    sin_diff = np.sin(phase_diff)
    cos_diff = np.cos(phase_diff)
    
    # Mean resultant length of phase differences
    psi = np.sqrt(np.mean(sin_diff)**2 + np.mean(cos_diff)**2)
    
    return psi

In [30]:
def calculate_coherence(signal_x, signal_y, fs, freq_range, nperseg=100, noverlap=None):
    low, high = freq_range[0], freq_range[1]
    """
    Calculate coherence between two signals
    
    Parameters:
    signal_x, signal_y: 1D arrays
    fs: sampling frequency
    nperseg: length of each segment
    noverlap: number of points to overlap
    
    Returns:
    frequencies: frequency points
    coherence: coherence values at each frequency
    """
    signals = np.stack([signal_x, signal_y], axis=0)
    csd = mne.time_frequency.csd_array_multitaper(signals.reshape(1,2,-1), 1000,  fmin=low, fmax=high, adaptive=True, low_bias=True)
    Pxy = csd.data
    f = csd.frequencies
    Pxx,f = mne.time_frequency.psd_array_multitaper(signal_x, 1000, fmin=low, fmax=high,adaptive=True, low_bias=True)
    Pyy,f  = mne.time_frequency.psd_array_multitaper(signal_y, 1000, fmin=low, fmax=high,adaptive=True, low_bias=True)

    coherence = np.abs(Pxy)**2 / (Pxx * Pyy)
    
    return f, coherence

check with mne.time_frequency.psd_arraay_multitaper can be applied to multiple signals simultaneously

def calculate_coherence(signal_x, signal_y, fs, nperseg=100, noverlap=None):
    """
    Calculate coherence between two signals
    
    Parameters:
    signal_x, signal_y: 1D arrays
    fs: sampling frequency
    nperseg: length of each segment
    noverlap: number of points to overlap
    
    Returns:
    frequencies: frequency points
    coherence: coherence values at each frequency
    """
    f, Pxy = signal.csd(signal_x, signal_y, fs, nperseg=nperseg, noverlap=noverlap)
    f, Pxx = signal.welch(signal_x, fs, nperseg=nperseg, noverlap=noverlap)
    f, Pyy = signal.welch(signal_y, fs, nperseg=nperseg, noverlap=noverlap)
    
    coherence = np.abs(Pxy)**2 / (Pxx * Pyy)
    
    return f, coherence

In [31]:
def load_data_set(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index =  subject_index
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    print(all_epochs.shape)
    
    return all_epochs, labels_raw, ch_names

In [32]:
def load_data_set_freq(freq_band, subject_index=2):
    cfg = load_config(CFG_YAML_freq)
    cfg.dataset.subject_index =  subject_index
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg,freq_band = freq_band)
    print(all_epochs.shape)
    
    return all_epochs, labels_raw, ch_names

In [33]:
freq_bands = {"delta" : (0.5, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}

find more efficient way or forget about it, as it is way too expensive right now

In [34]:
def build_dataframe_subject_pac(subject_index=2):
    df = pd.DataFrame(columns=['subject_index', 'trial_index', 'ch_index', 'ch_name', 'freq_band1', 'freq_band2', 'mi'])
    freq_band_pairs = [(band1, band2) for i, band1 in enumerate(freq_bands.keys()) for j, band2 in enumerate(freq_bands.keys()) if i < j]
    i = 0
    for pair in freq_band_pairs:
        all_epochs_freq1, _, ch_names_freq1 = load_data_set_freq(pair[0], subject_index)
        all_epochs_freq2, _, _ = load_data_set_freq(pair[1], subject_index)
        for trial_idx in range(all_epochs_freq1.shape[0]):
            for ch_idx in range(all_epochs_freq1.shape[1]):
                signal_x = all_epochs_freq1[trial_idx, ch_idx, :]
                signal_y = all_epochs_freq2[trial_idx, ch_idx, :]
                mi = calculate_modulation_index(signal_x, signal_y)
                df.loc[i] = [subject_index, trial_idx, ch_idx, ch_names_freq1[ch_idx], pair[0], pair[1], mi]
                #print(df["mi"])
                i += 1

    return df



In [35]:
def build_dataframe_subject_coherence(subject_index=2):
    df = pd.DataFrame(columns=['subject_index', 'trial_index', 'ch_index1', 'ch_index2', 'ch_name1', 'ch_name2' 'coherence'])
    all_epochs, _, ch_names = load_data_set(subject_index)
    i = 0
    #CSDs = mne.time_frequency.csd_array_multitaper(all_epochs, 1000,  fmin=0, fmax=45, adaptive=True, low_bias=True)
    #PSDs1 = mne.time_frequency.psd_array_multitaper(all_epochs, 1000, adaptive=True, low_bias=True)
    #PSDs2 = mne.time_frequency.psd_array_multitaper(all_epochs, 1000, adaptive=True, low_bias=True)
    for freq_band, freq_range in freq_bands.items():
        for trial_idx in range(all_epochs.shape[0]):
            for ch_idx1 in range(all_epochs.shape[1]):
                for ch_idx2 in range(all_epochs.shape[1]):
                    if ch_idx1 < ch_idx2:
                        signal_x = all_epochs[trial_idx, ch_idx1, :]
                        signal_y = all_epochs[trial_idx, ch_idx2, :]
                        f, coherence = calculate_coherence(signal_x, signal_y, 1000, freq_range)
                        df.loc[i] = [subject_index, trial_idx, ch_idx1, ch_idx2, ch_names[ch_idx1], ch_names[ch_idx2], coherence]
                        i += 1

In [36]:
def build_dataframe_subject_psi(subject_index=2):
    df = pd.DataFrame(columns=['subject_index', 'trial_index', 'ch_index1', 'ch_index2', 'ch_name', 'psi'])
    i = 0
    for freq_band, freq_range in freq_bands.items():
        all_epochs, _, ch_names = load_data_set_freq(freq_band, subject_index)
        for trial_idx in range(all_epochs.shape[0]):
            for ch_idx1 in range(all_epochs.shape[1]):
                for ch_idx2 in range(all_epochs.shape[1]):
                    if ch_idx1 < ch_idx2:
                        psi = calculate_psi(all_epochs[trial_idx, ch_idx1, :], all_epochs[trial_idx, ch_idx2, :], 1000, freq_range)
                        df.loc[i] = [subject_index, trial_idx, ch_idx1, ch_idx2, ch_names[ch_idx1], psi]
                        i += 1

In [37]:
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    df = build_dataframe_subject_pac(subject_index)
    os.makedirs("mi", exist_ok=True)
    df.to_csv(f"mi/modulation_index_subject_{subject_index}.csv")


Loading EEG data...

subject index: 1, frequency band: delta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_delta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_delta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(510, 60, 900)
Loading EEG data...

subject index: 1, frequency band: theta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_theta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_theta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(510, 60, 900)


...

(100, 60, 900)
(1, 60, 1)
(510, 60, 900)
Loading EEG data...

subject index: 1, frequency band: beta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_beta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
510 matching events found


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_beta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(510, 60, 900)


/home/marco/anaconda3/envs/mne/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/marco/anaconda3/envs/mne/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


KeyboardInterrupt: 

In [38]:
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    df = build_dataframe_subject_psi(subject_index)
    os.makedirs("psi", exist_ok=True)
    df.to_csv(f"psi/psi_subject_{subject_index}.csv")
    

Loading EEG data...

subject index: 1, frequency band: delta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_delta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_delta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(510, 60, 900)
Loading EEG data...

subject index: 1, frequency band: theta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_theta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_theta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(510, 60, 900)
Loading EEG data...

subject index: 1, frequency band: alpha

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_alpha_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_alpha_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


(100, 60, 900)
(1, 60, 1)
(510, 60, 900)
Loading EEG data...

subject index: 1, frequency band: beta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_beta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_001_bandpass_beta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


(510, 60, 900)


KeyboardInterrupt: 